In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Zomato_Dataset.csv")

### Transformação das colunas latitudes e longitudes dos restaurantes e clientes

In [3]:
# Cria colunas corrigidas (mantém as originais intactas, boa prática)
df["Restaurant_latitude_fix"] = df["Restaurant_latitude"].abs()
df["Restaurant_longitude_fix"] = df["Restaurant_longitude"].abs()
df["Delivery_location_latitude_fix"] = df["Delivery_location_latitude"].abs()
df["Delivery_location_longitude_fix"] = df["Delivery_location_longitude"].abs()

# mesmo depois da correção de sinal — esses não dá pra "consertar", só descartar
LAT_MIN, LAT_MAX = 6.0, 38.0
LON_MIN, LON_MAX = 68.0, 98.0

for col_lat, col_lon in [
    ("Restaurant_latitude_fix", "Restaurant_longitude_fix"),
    ("Delivery_location_latitude_fix", "Delivery_location_longitude_fix"),
]:
    fora_do_range = ~(
        df[col_lat].between(LAT_MIN, LAT_MAX) &
        df[col_lon].between(LON_MIN, LON_MAX)
    )
    print(f"{col_lat}: {fora_do_range.sum()} pontos ainda inválidos após correção")
    df.loc[fora_do_range, [col_lat, col_lon]] = np.nan

Restaurant_latitude_fix: 3640 pontos ainda inválidos após correção
Delivery_location_latitude_fix: 3640 pontos ainda inválidos após correção


In [4]:
# Excluindo valores NA das latitudes e longitudes

df = df.dropna(subset=["Restaurant_latitude_fix"])
df = df.dropna(subset=["Restaurant_longitude_fix"])
df = df.dropna(subset=["Delivery_location_latitude_fix"])
df = df.dropna(subset=["Delivery_location_longitude_fix"])

### Transformação da coluna Delivery_person_ID

In [5]:
# Transformando os valores nulos com a mediana

mediana_idade = df["Delivery_person_Age"].median() # Calculando a mediana

df.loc[
    df["Delivery_person_Age"].isna(),
    "Delivery_person_Age"
] = mediana_idade

In [6]:
# Aplicando a mediana para idade de motoristas abaixo de 18 anos

df.loc[
    df["Delivery_person_Age"] < 18,
    "Delivery_person_Age"
] = mediana_idade

### Transformação da coluna Delivery_person_Ratings

In [7]:
# Alterando valores acima de 5, aplicando o valor máximo

df.loc[
    df["Delivery_person_Ratings"] > 5,
    "Delivery_person_Ratings"
] = 5

In [8]:
# Aplicando a mediana nos valores nulos

mediana_notas = df["Delivery_person_Ratings"].median()

df.loc[
    df["Delivery_person_Ratings"].isnull(),
    "Delivery_person_Ratings"
] = mediana_notas

### Transformação da coluna Delivery_person_Age

In [9]:
print(df["Delivery_person_Age"].min())
print(df["Delivery_person_Age"].max())

df["Delivery_person_Age"] = df["Delivery_person_Age"].astype(int)

20.0
50.0


### Transformação da coluna Order_Date

In [10]:
# Transformando a data de OBJECT para DATETIME

df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="%d-%m-%Y"
)

df["Order_Date"].dtype

dtype('<M8[ns]')

### Transformação da coluna Time_Orderd

In [11]:
def converter_hora(valor):
    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip()

    if ":" in valor:
        hora = pd.to_datetime(valor, format="%H:%M", errors="coerce")
        if pd.isna(hora):
            return pd.NA
        return hora.strftime("%H:%M")

    try:
        hora = pd.to_datetime(float(valor), unit="D", origin="1899-12-30")
        if pd.isna(hora):
            return pd.NA
        return hora.round("min").strftime("%H:%M")
    except (ValueError, TypeError):
        return pd.NA


df["Time_Orderd"] = df["Time_Orderd"].apply(converter_hora)
df["Time_Orderd"] = pd.to_timedelta(df["Time_Orderd"].astype(str) + ":00", errors="coerce")



In [12]:
df["Time_Orderd"].head()

0   0 days 21:55:00
1   0 days 14:55:00
2   0 days 17:30:00
3   0 days 09:20:00
4   0 days 19:50:00
Name: Time_Orderd, dtype: timedelta64[ns]

In [13]:
# Deletando os valores nulos visto que é uma coluna MUITO IMPORTANTE para análise

df = df.dropna(subset=["Time_Orderd"])

### Transformação da coluna Time_Order_picked

In [14]:
# Transformando o horário de OBJECT para TIME

def converter_hora_picked(valor):
    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip()

    if ":" in valor:
        hora = pd.to_datetime(valor, format="%H:%M", errors="coerce")
        if pd.isna(hora):
            return pd.NA
        return hora.strftime("%H:%M")

    try:
        hora = pd.to_datetime(float(valor), unit="D", origin="1899-12-30")
        if pd.isna(hora):
            return pd.NA
        return hora.round("min").strftime("%H:%M")
    except (ValueError, TypeError):
        return pd.NA


df["Time_Order_picked"] = df["Time_Order_picked"].apply(converter_hora_picked)
df["Time_Order_picked"] = pd.to_timedelta(df["Time_Order_picked"].astype(str) + ":00", errors="coerce")

In [15]:
df["Time_Order_picked"].head()

0   0 days 22:10:00
1   0 days 15:05:00
2   0 days 17:40:00
3   0 days 09:30:00
4   0 days 20:05:00
Name: Time_Order_picked, dtype: timedelta64[ns]

In [16]:
# Deletando os valores nulos visto que é uma coluna MUITO IMPORTANTE para análise

df = df.dropna(subset=["Time_Order_picked"])

### Transformação da coluna multiple_deliveries

In [17]:
# Aplicando a mediana para valores nulos

mediana_multiplos_delivery = df["multiple_deliveries"].median()

df.loc[
    df["multiple_deliveries"].isna(),
    "multiple_deliveries"
] = mediana_multiplos_delivery

In [18]:
# Convertendo valores FLOAT para INT

df["multiple_deliveries"] = df["multiple_deliveries"].astype(int)

### Criando a coluna dia_semana

In [19]:
dias = {
    "Monday": "Segunda-feira",
    "Tuesday": "Terça-feira",
    "Wednesday": "Quarta-feira",
    "Thursday": "Quinta-feira",
    "Friday": "Sexta-feira",
    "Saturday": "Sábado",
    "Sunday": "Domingo"
}

df["dia_semana"] = df["Order_Date"].dt.day_name().map(dias)

### Criando a coluna Faixa_Horario_Pedido

In [20]:
# Criando a coluna faixa de horário de acordo com o horário do pedido

def faixa_horario(valor):
    if pd.isna(valor):
        return pd.NA

    hora = valor.components.hours  # extrai só a hora (0-23) do timedelta

    if 5 <= hora < 11:
        return "Manhã"
    elif 11 <= hora < 14:
        return "Almoço"
    elif 14 <= hora < 18:
        return "Tarde"
    elif 18 <= hora < 21:
        return "Jantar"
    else:  # 21h-23h e 0h-4h
        return "Noite"

df["Faixa_Horario_Pedido"] = df["Time_Orderd"].apply(faixa_horario)

df["Faixa_Horario_Pedido"].head()

0     Noite
1     Tarde
2     Tarde
3     Manhã
4    Jantar
Name: Faixa_Horario_Pedido, dtype: object

### Criando a coluna Tempo_Coleta_Min

In [21]:
# Criando a coluna tempo de coleta que é o horário de coleta do pedido subtraído pelo horário do pedido

df["Tempo_Coleta_Min"] = df["Time_Order_picked"] - df["Time_Orderd"]

df["Tempo_Coleta_Min"].head(10)

0   0 days 00:15:00
1   0 days 00:10:00
2   0 days 00:10:00
3   0 days 00:10:00
4   0 days 00:15:00
5   0 days 00:10:00
6   0 days 00:15:00
7   0 days 00:10:00
8   0 days 00:10:00
9   0 days 00:15:00
Name: Tempo_Coleta_Min, dtype: timedelta64[ns]

In [22]:
# Convertendo datas negativas devido a pedidos no dia e entrega no próximo dia, na virada de horário da meia noite

df["Tempo_Coleta_Min"] = df["Tempo_Coleta_Min"].apply(
    lambda x: x + pd.Timedelta(days=1) if x < pd.Timedelta(0) else x
)

In [23]:
# Transformando em minutos

df["Tempo_Coleta_Min"] = df["Tempo_Coleta_Min"].dt.total_seconds() / 60

### Criando a coluna distancia_km

In [24]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # raio da Terra em km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df["distancia_km"] = haversine(
    df["Restaurant_latitude_fix"], df["Restaurant_longitude_fix"],
    df["Delivery_location_latitude_fix"], df["Delivery_location_longitude_fix"]
)

print(df["distancia_km"].describe())
print("Linhas com distância NaN (coordenada inválida):", df["distancia_km"].isna().sum())

df["distancia_km"] = df["distancia_km"].round(2)

count    39569.000000
mean         9.670997
std          5.602958
min          1.465067
25%          4.657569
50%          9.192290
75%         13.631331
max         20.969489
Name: distancia_km, dtype: float64
Linhas com distância NaN (coordenada inválida): 0


In [25]:
df["distancia_km"].head()

0    10.28
1     6.24
2    13.79
3     2.93
4    19.40
Name: distancia_km, dtype: float64

### Criando a coluna Faixa_Etaria_Entregador

In [26]:
# Criando a coluna faixa etaria do entregador de acordo com sua idade (18-25, 26-35, 36-45, 46+)

def faixa_etaria(idade):
    if pd.isna(idade):
        return pd.NA


    if idade < 25:
        return "jovem adulto"
    elif idade >= 25 and idade <= 35:
        return "adulto"
    elif idade >= 36 and idade <= 50:
        return "meia idade"
    else:
        "idoso"


df["Faixa_Etaria_Entregador"] = df["Delivery_person_Age"].apply(faixa_etaria)

df["Faixa_Etaria_Entregador"].head()

0      meia idade
1    jovem adulto
2    jovem adulto
3          adulto
4    jovem adulto
Name: Faixa_Etaria_Entregador, dtype: object

In [27]:
df["Faixa_Etaria_Entregador"].unique()

array(['meia idade', 'jovem adulto', 'adulto'], dtype=object)

### Criando a coluna Faixa_Avaliacao

In [28]:
# Criando a coluna faixa avaliação do entregador de acordo com sua nota (abaixo de 4.0, 4.0-4.5, acima de 4.5)

def faixa_avaliacao(nota):
    if pd.isna(nota):
        return pd.NA
    
    if nota <= 3.5:
        return "ruim"
    elif nota > 3.5 and nota < 4.5:
        return "média"
    else:
        return "boa"
    

df["Faixa_Avaliacao"] = df["Delivery_person_Ratings"].apply(faixa_avaliacao)

df["Faixa_Avaliacao"].head() 


0    média
1      boa
2      boa
3    média
4      boa
Name: Faixa_Avaliacao, dtype: object

In [29]:
df["Faixa_Avaliacao"].unique()

array(['média', 'boa', 'ruim'], dtype=object)

### Criando a coluna Dentro_SLA

In [30]:
# Criando a coluna Dentro_SLA

def calcular_sla(distancia):
    if distancia <= 3:
        return 15
    elif distancia <= 6:
        return 20
    elif distancia <= 10:
        return 30
    else:
        return 50

df["SLA"] = df["distancia_km"].apply(calcular_sla)

df["Dentro_SLA"] = np.where(
    df["Time_taken (min)"] <= df["SLA"],
    "Sim",
    "Não"
)

df["Dentro_SLA"].head()

0    Sim
1    Sim
2    Sim
3    Não
4    Sim
Name: Dentro_SLA, dtype: object

### Criando a coluna Combinacao_Clima_Transito

In [31]:
# Traduzindo a coluna 'Weather_conditions' antes de criar a coluna Combinacao_Clima_Transito 

traducao_clima = {
    "Fog": "Neblina",
    "Stormy": "Tempestade",
    "Sandstorms": "Tempestade de areia",
    "Windy": "Ventando",
    "Cloudy": "Nublado",
    "Sunny": "Ensolarado"
}

df["Weather_conditions"] = df["Weather_conditions"].replace(traducao_clima)

df["Weather_conditions"].head()


0                Neblina
1             Tempestade
2    Tempestade de areia
3    Tempestade de areia
4                Neblina
Name: Weather_conditions, dtype: object

In [32]:
# Traduzindo a coluna 'Road_traffic_density' antes de criar a coluna Combinacao_Clima_Transito 

traducao_transito = {
    "Jam": "Congestionamento",
    "High": "Alto",
    "Medium": "Médio",
    "Low": "Baixo"
}

df["Road_traffic_density"] = df["Road_traffic_density"].replace(traducao_transito)

df["Road_traffic_density"].head()

0    Congestionamento
1                Alto
2               Médio
3               Baixo
4    Congestionamento
Name: Road_traffic_density, dtype: object

In [33]:
# Criando a coluna Combinacao_Clima_Transito 

df["Combinacao_Clima_Transito"] = (
    df["Weather_conditions"].fillna("Desconhecido").str.strip()
    + " | "
    + df["Road_traffic_density"].fillna("Desconhecido").str.strip()
)

df["Combinacao_Clima_Transito"].head()

0     Neblina | Congestionamento
1              Tempestade | Alto
2    Tempestade de areia | Médio
3    Tempestade de areia | Baixo
4     Neblina | Congestionamento
Name: Combinacao_Clima_Transito, dtype: object

### Traduzindo e Reogarnizando as colunas

In [47]:
df = df.rename(columns={

    # Identificadores
    "ID": "ID_Pedido",
    "Delivery_person_ID": "ID_Entregador",

    # Entregador
    "Delivery_person_Age": "Idade_Entregador",
    "Delivery_person_Ratings": "Avaliacao_Entregador",

    # Restaurante
    "Restaurant_latitude": "Latitude_Restaurante",
    "Restaurant_longitude": "Longitude_Restaurante",

    # Entrega
    "Delivery_location_latitude": "Latitude_Entrega",
    "Delivery_location_longitude": "Longitude_Entrega",

    # Datas e horários
    "Order_Date": "Data_Pedido",
    "Time_Orderd": "Horario_Pedido",
    "Time_Order_picked": "Horario_Coleta",

    # Condições externas
    "Weather_conditions": "Condicao_Climatica",
    "Road_traffic_density": "Densidade_Transito",

    # Pedido
    "Vehicle_condition": "Condicao_Veiculo",
    "Type_of_order": "Tipo_Pedido",
    "Type_of_vehicle": "Tipo_Veiculo",
    "multiple_deliveries": "Entregas_Simultaneas",
    "Festival": "Festival",
    "City": "Cidade",
    "Time_taken (min)": "Tempo_Entrega_Min",

    # Coordenadas tratadas
    "Restaurant_latitude_fix": "Latitude_Restaurante_Fix",
    "Restaurant_longitude_fix": "Longitude_Restaurante_Fix",
    "Delivery_location_latitude_fix": "Latitude_Entrega_Fix",
    "Delivery_location_longitude_fix": "Longitude_Entrega_Fix",

    # Colunas calculadas
    "dia_semana": "Dia_Semana",
    "distancia_km": "Distancia_Km"
})

# Reoganizando

df = df[

    # Identificadores
    [
        "ID_Pedido",
        "ID_Entregador",

        # Entregador
        "Idade_Entregador",
        "Faixa_Etaria_Entregador",
        "Avaliacao_Entregador",
        "Faixa_Avaliacao",

        # Restaurante
        "Latitude_Restaurante",
        "Longitude_Restaurante",
        "Latitude_Restaurante_Fix",
        "Longitude_Restaurante_Fix",

        # Entrega
        "Latitude_Entrega",
        "Longitude_Entrega",
        "Latitude_Entrega_Fix",
        "Longitude_Entrega_Fix",
        "Distancia_Km",

        # Datas e horários
        "Data_Pedido",
        "Dia_Semana",
        "Horario_Pedido",
        "Faixa_Horario_Pedido",
        "Horario_Coleta",
        "Tempo_Coleta_Min",

        # Tempo da entrega
        "Tempo_Entrega_Min",
        "SLA",
        "Dentro_SLA",

        # Condições externas
        "Condicao_Climatica",
        "Densidade_Transito",
        "Combinacao_Clima_Transito",

        # Pedido
        "Condicao_Veiculo",
        "Tipo_Pedido",
        "Tipo_Veiculo",
        "Entregas_Simultaneas",
        "Festival",
        "Cidade"
    ]

]

### Tradução das linhas

In [51]:
# Tradução da cidade

traducao_cidade = {
    "Metropolitian": "Metropolitana",
    "Urban": "Urbana",
    "Semi-Urban": "Semiurbana"
}

df["Cidade"] = df["Cidade"].replace(traducao_cidade)

In [52]:
# Tradução do Festival

traducao_festival = {
    "Yes": "Sim",
    "No": "Não"
}

df["Festival"] = df["Festival"].replace(traducao_festival)

In [53]:
# Tradução do Tipo de Veiculo

traducao_veiculo = {
    "motorcycle": "Motocicleta",
    "scooter": "Scooter",
    "electric_scooter": "Scooter Elétrica",
    "bicycle": "Bicicleta"
}

df["Tipo_Veiculo"] = df["Tipo_Veiculo"].replace(traducao_veiculo)

In [54]:
# Tradução do Tipo de Pedido

traducao_pedido = {
    "Snack": "Lanche",
    "Meal": "Refeição",
    "Drinks": "Bebidas",
    "Buffet": "Buffet"
}

df["Tipo_Pedido"] = df["Tipo_Pedido"].replace(traducao_pedido)

### Salvando o relatório

In [56]:
df.to_csv(
    r"C:\Users\Mathe\Desktop\Estudos\Projetos\01 - Zomato\Analise-Dados-Zomato\data\processed\zomato_tratado.csv",
    index=False,
    encoding="utf-8-sig"
)